# 01 — Làm sạch, lấy tất cả Attack, chia Dirichlet và fit scaler

Toàn bộ CSV → loại nhóm có cùng 39 predictor nhưng khác nhãn binary → giữ dòng nguồn đầu tiên của mỗi input → giữ tất cả Attack sạch → lấy Benign không hoàn lại cho đủ tối đa 1,5 triệu mẫu. Không cân bằng 50–50.

Ranh giới fit/validation/test được quyết định bằng hash input với tỷ lệ mục tiêu 63/7/30. Sau đó cùng một bộ tỷ lệ Dirichlet cho mỗi nhóm metadata được áp dụng trong cả ba split để phân phối mẫu cho client. Tỷ lệ và số mẫu client là kết quả ngẫu nhiên của seed cố định; có làm tròn số nguyên. Nhóm hiếm có thể vắng ở một client/split. Scaler fit lại chỉ từ fit của client đó. CSV nguồn không bị sửa.

In [1]:
from pathlib import Path
import os, sys, json, importlib
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p/'COPY_PROVENANCE.json').is_file()
               and (p/'configs/tuning.json').is_file() and (p/'notebook_workflow.py').is_file())
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for module_name in ('common', 'feature_preprocessing', 'prepare', 'models', 'metrics', 'notebook_workflow'):
    module = importlib.import_module(module_name)
    assert Path(module.__file__).resolve().parent == PROJECT.resolve(), 'Restart kernel to load this copy.'
    importlib.reload(module)
workflow = sys.modules['notebook_workflow']
CONFIG = PROJECT/'configs/tuning.json'
C = workflow.settings(CONFIG)
RUN_ROOT = workflow.run_root(C)
print('Project:', PROJECT)
print('Results:', RUN_ROOT)
display(workflow.budget(CONFIG))

Project: .
Results: runs/cic_all_attack_dirichlet_a05_max1p5m


{'tasks': ['binary'],
 'tuning_task': 'binary',
 'models': ['dnn', 'cnn'],
 'trials_per_model': 10,
 'rounds': 15,
 'local_epochs': 3,
 'baseline_epochs': 45,
 'client_optimizer': 'adam',
 'server_optimizer': 'sgd',
 'server_learning_rate': 1.0,
 'tuning_weighting': 'number_of_examples',
 'final_weightings': ['number_of_examples'],
 'selection': 'mean_cross_client_validation_f1',
 'checkpoint_selection': 'final_round',
 'total_tuning_trials': 20,
 'fits_per_trial': '3 Local + 1 Centralized + 1 sample-weighted FedAvg',
 'objective': 'mean of 12 validation F1: 6 cross-client Local + 3 Centralized + 3 FL',
 'trial_1': {'dnn': {'architecture': '8-4-2',
   'dropout': 0.5,
   'batch_size': 1024,
   'client_learning_rate': 0.0005},
  'cnn': {'architecture': '8-16-8',
   'kernel_size': 3,
   'dropout': 0.2,
   'batch_size': 2048,
   'client_learning_rate': 0.0005}}}

In [2]:
manifest = workflow.prepare_validation(CONFIG)
display(manifest["source"]["cleaning"])
display(manifest["selection"])
display(manifest["audit"])
frame = workflow.split_table(CONFIG)
display(frame[["client","split","rows","benign","attack","attack_fraction"]])

Reusing verified prepared data: runs/cic_all_attack_dirichlet_a05_max1p5m/prepared


{'raw_rows': 18893708,
 'finite_rows': 18893708,
 'excluded_nonfinite': 0,
 'binary_conflict_groups_removed': 0,
 'binary_conflict_rows_removed': 0,
 'duplicate_rows_removed': 9912634,
 'clean_rows': 8981074,
 'clean_benign': 8436211,
 'clean_attack': 544863,
 'groups_with_multiple_original_attack_labels_retained': 1991,
 'representative_family_counts': [8436211,
  14720,
  472,
  338755,
  114482,
  76012,
  422],
 'representative_attack_counts': [8436211,
  472,
  190,
  165,
  95584,
  2107,
  16791,
  27018,
  311534,
  15,
  188,
  19,
  76012,
  67,
  14701]}

{'method': 'all_clean_attack_then_benign_without_replacement',
 'max_pool_rows': 1500000,
 'source_clean_rows': 8981074,
 'source_clean_attack': 544863,
 'source_clean_benign': 8436211,
 'selected_rows': 1500000,
 'selected_attack': 544863,
 'selected_benign': 955137,
 'attack_fraction': 0.363242,
 'all_clean_attack_rows_selected': True,
 'all_clean_attack_row_ids_sha256': 'e58735ab8af4fa26303c99715a92d650a990849a42a11e68e877c9df430f1bcd',
 'selected_original_attack_counts': [955137,
  472,
  190,
  165,
  95584,
  2107,
  16791,
  27018,
  311534,
  15,
  188,
  19,
  76012,
  67,
  14701]}

{'selected_rows': 1500000,
 'unique_row_ids': 1500000,
 'shared_input_groups_between_any_client_or_split': 0,
 'unique_selected_input_groups': 1500000,
 'duplicate_input_rows': 0}

,client,split,rows,benign,attack,attack_fraction
0,cic_1,fit,308370,231422,76948,0.249531
1,cic_1,validation,34111,25779,8332,0.244261
2,cic_1,test,146314,109815,36499,0.249457
3,cic_2,fit,582052,368289,213763,0.367258
4,cic_2,validation,64620,41024,23596,0.365150
5,cic_2,test,276449,174762,101687,0.367833
6,cic_3,fit,55584,2551,53033,0.954105
7,cic_3,validation,6104,284,5820,0.953473
8,cic_3,test,26396,1211,25185,0.954122


In [3]:
import pandas as pd
from common import CLASS_NAMES
shares = pd.DataFrame(manifest["partition"]["sampled_group_client_shares"], index=CLASS_NAMES, columns=manifest["partition"]["clients"])
print("Mỗi hàng: phần của một nhóm giao cho từng client, tổng = 100%.")
display(shares.style.format("{:.2%}"))
counts = pd.DataFrame(frame["class_counts"].tolist(), columns=CLASS_NAMES)
display(pd.concat([frame[["client","split"]].reset_index(drop=True),counts],axis=1))
display(frame.groupby("client")[["rows","benign","attack"]].sum())

Mỗi hàng: phần của một nhóm giao cho từng client, tổng = 100%.


,cic_1,cic_2,cic_3
Benign,38.43%,61.15%,0.42%
BruteForce,83.08%,1.80%,15.11%
Bot,24.56%,17.04%,58.39%
DoS,14.19%,70.14%,15.67%
DDoS,2.87%,84.33%,12.80%
Infiltration,76.33%,5.84%,17.83%
WebAttack,10.29%,30.77%,58.94%


,client,split,Benign,BruteForce,Bot,DoS,DDoS,Infiltration,WebAttack
0,cic_1,fit,231422,7752,75,30348,2066,36679,28
1,cic_1,validation,25779,866,11,3329,232,3891,3
2,cic_1,test,109815,3612,31,14403,989,17452,12
3,cic_2,fit,368289,168,52,149969,60682,2808,84
4,cic_2,validation,41024,19,7,16448,6814,298,10
5,cic_2,test,174762,78,21,71174,29042,1336,36
6,cic_3,fit,2551,1410,178,33507,9213,8565,160
7,cic_3,validation,284,158,25,3675,1035,908,19
8,cic_3,test,1211,657,72,15902,4409,4075,70


,rows,benign,attack
client,,,
cic_1,488795,367016,121779
cic_2,923121,584075,339046
cic_3,88084,4046,84038


In [4]:
import numpy as np
rows = []
vocabulary = manifest["source"]["attack_vocabulary"]
for client in manifest["datasets"]:
    for split in ("fit","validation","test"):
        with np.load(RUN_ROOT/"prepared"/client/f"{split}_meta.npz") as z:
            n = np.bincount(z["attack_code"], minlength=len(vocabulary))
        rows.extend({"client":client,"split":split,"attack":label,"samples":int(n[i])} for i,label in enumerate(vocabulary))
support = pd.DataFrame(rows)
display(support.pivot(index="attack",columns=["client","split"],values="samples").fillna(0).astype(int))
display(support.groupby("attack")["samples"].sum().to_frame("all_clients_all_splits"))

client                     cic_1                      cic_2             \
split                        fit validation    test     fit validation   
attack                                                                   
Benign                    231422      25779  109815  368289      41024   
Bot                           75         11      31      52          7   
Brute Force -Web               8          2       5      34          5   
Brute Force -XSS              14          1       4      39          3   
DDOS attack-HOIC            1725        195     810   50628       5731   
DDOS attack-LOIC-UDP          39          8      24    1097        132   
DDoS attacks-LOIC-HTTP       302         29     155    8957        951   
DoS attacks-GoldenEye       2420        261    1161   12036       1249   
DoS attacks-Hulk           27907       3068   13232  137851      15192   
DoS attacks-SlowHTTPTest       1          0       1       7          1   
DoS attacks-Slowloris         20          0       9      75          6   
FTP-BruteForce                10          1       3       1          0   
Infilteration              36679       3891   17452    2808        298   
SQL Injection                  6          0       3      11          2   
SSH-Bruteforce              7742        865    3609     167         19   

client                            cic_3                    
split                       test    fit validation   test  
attack                                                     
Benign                    174762   2551        284   1211  
Bot                           21    178         25     72  
Brute Force -Web              15     76          9     36  
Brute Force -XSS              17     57          7     23  
DDOS attack-HOIC           24224   7740        852   3679  
DDOS attack-LOIC-UDP         533    178         19     77  
DDoS attacks-LOIC-HTTP      4285   1295        164    653  
DoS attacks-GoldenEye       5694   2680        282   1235  
DoS attacks-Hulk           65431  30801       3390  14662  
DoS attacks-SlowHTTPTest       2      2          0      1  
DoS attacks-Slowloris         47     24          3      4  
FTP-BruteForce                 0      1          0      3  
Infilteration               1336   8565        908   4075  
SQL Injection                  4     27          3     11  
SSH-Bruteforce                78   1409        158    654

,all_clients_all_splits
attack,
Benign,955137
Bot,472
Brute Force -Web,190
Brute Force -XSS,165
DDOS attack-HOIC,95584
DDOS attack-LOIC-UDP,2107
DDoS attacks-LOIC-HTTP,16791
DoS attacks-GoldenEye,27018
DoS attacks-Hulk,311534


In [5]:
display(pd.DataFrame({"weighted":workflow.client_weights(C,"number_of_examples")}))
for client, info in manifest["datasets"].items():
    print(client,"scaler n_fit:",info["scaler"]["n_fit"])
    assert info["scaler"]["n_fit"] == info["fit_rows"]

,weighted
cic_1,0.325970
cic_2,0.615273
cic_3,0.058756


cic_1 scaler n_fit: 308370
cic_2 scaler n_fit: 582052
cic_3 scaler n_fit: 55584


Với CSV đã xác minh: 544.863 Attack sạch và 955.137 Benign, tổng 1.500.000; tỷ lệ Attack toàn pool khoảng 36,32%. Đây là lựa chọn giữ hết Attack, không phải tỷ lệ nguồn sạch tự nhiên 6,07%. Metadata giữ cả nhãn Attack đại diện và tập nhãn nguồn để báo cáo phần đa nghĩa. Tuning chỉ dùng fit/validation.